# Machine Learning Pipeline for Binary Target

The notebook is divided into small cells so each method can be selected, changed, or skipped independently.

## 1. Import Libraries

Core libraries for data preparation, modeling, validation, tuning, and interpretation.

**Purpose:** Load the tools used throughout the workflow so later cells can prepare data, train models, and evaluate predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

## 2. Load Data

The default code assumes `train.csv` contains labelled rows and `test.csv` contains unlabelled rows. If the labelled data is split between `train.csv` and `test.csv`, while a separate `data_without_label.csv` is used for predictions, replace the two default file-loading lines with the commented `pd.concat` block below. Both files being combined must contain the target column.

**Purpose:** Load the standard labelled and unlabelled files, or uncomment the provided concatenation snippet when the labelled rows are split across two files.

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("submission_template.csv")

# If the labelled rows are split across train.csv and test.csv, and the
# separate prediction file is data_without_label.csv, replace the two
# assignments above with this block:
# train = pd.concat(
#     [pd.read_csv("train.csv"), pd.read_csv("test.csv")],
#     ignore_index=True
# )
# test = pd.read_csv("data_without_label.csv")

In [ ]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", submission.shape)

display(train.head())

## 3. Quick Data Check

**Purpose:** Inspect the dataset's structure and basic quality before making assumptions about columns or preprocessing.

In [ ]:
train.info()

In [ ]:
data_check = pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique": train.nunique(dropna=False)
})

display(data_check)

In [ ]:
print("Duplicate rows:", train.duplicated().sum())
display(train.describe(include="all").T)

## 4. Target Definition

Bad debt is defined as debt payment overdue for more than three months. Label 1 represents bad debt and label 0 represents no bad debt.

**Purpose:** Confirm what the positive and negative labels mean before selecting features or comparing models.

### Label Creation from Overdue Months

This transformation applies when the released data contains an overdue-duration field measured in months and no target label is supplied.

**Purpose:** Create a binary label from an overdue-duration field only when the released data does not already provide the target.

In [ ]:
# More than 3 months overdue = bad debt
train["bad_debt"] = (train["overdue_months"] > 3).astype(int)

train["bad_debt"].value_counts()

## 5. Select Target and Features

**Purpose:** Separate the target from input columns and exclude identifiers or fields that should not be used for prediction.

In [ ]:
target_col = "target"
id_cols = ["ID"]
drop_cols = []

In [ ]:
print(train[target_col].value_counts())
print()
print((train[target_col].value_counts(normalize=True) * 100).round(2))

In [ ]:
X = train.drop(columns=[target_col] + id_cols + drop_cols)
y = train[target_col]

X_test = test.drop(columns=id_cols + drop_cols)

print(X.shape, y.shape, X_test.shape)

## 6. Select Numerical and Categorical Columns

**Purpose:** Group input columns by data type so the preprocessing steps can be applied appropriately.

In [ ]:
numerical_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

## 7. Data Quality Tests

These checks identify structural issues that can affect preprocessing, validation, and model stability.

**Purpose:** Look for unusual structure or values that may require a documented data-preparation decision.

### Missingness, Cardinality, and Constant Features

High missingness, constant variables, and very high-cardinality variables can reduce model quality or create unnecessary complexity.

**Purpose:** Find columns with extensive missing data, little variation, or many distinct values that may complicate modelling.

In [ ]:
feature_check = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_pct": (X.isna().mean() * 100).round(2),
    "unique": X.nunique(dropna=False),
    "unique_ratio": (X.nunique(dropna=False) / len(X)).round(4)
})

display(feature_check.sort_values("missing_pct", ascending=False))

In [ ]:
constant_features = feature_check[feature_check["unique"] <= 1]
high_cardinality = feature_check[feature_check["unique_ratio"] >= 0.80]

print("Constant features")
display(constant_features)

print("High-cardinality features")
display(high_cardinality)

### Infinite Values

Positive or negative infinity often appears after ratio or log transformations and should be handled before model fitting.

**Purpose:** Check for infinite numeric values that can cause preprocessing or model-fitting failures.

In [ ]:
numeric_data = X.select_dtypes(include="number")

infinite_count = pd.DataFrame({
    "positive_inf": np.isposinf(numeric_data).sum(),
    "negative_inf": np.isneginf(numeric_data).sum()
})

display(infinite_count[
    (infinite_count["positive_inf"] > 0) |
    (infinite_count["negative_inf"] > 0)
])

### Target Imbalance

A large difference between class frequencies can make accuracy misleading and may require class weighting, threshold adjustment, or resampling.

**Purpose:** Measure class proportions to understand whether the evaluation and training setup should account for unequal class counts.

In [ ]:
target_distribution = pd.DataFrame({
    "count": y.value_counts(),
    "percent": (y.value_counts(normalize=True) * 100).round(2)
})

display(target_distribution)

## 8. Outlier Tests

The IQR rule flags observations below Q1 - 1.5×IQR or above Q3 + 1.5×IQR. Outliers are diagnostic signals and are not automatically errors.

**Purpose:** Flag unusually extreme numeric observations for review; this diagnostic does not by itself justify deleting or changing them.

In [ ]:
outlier_rows = []

for col in numerical_cols:
    q1 = X[col].quantile(0.25)
    q3 = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = ((X[col] < lower) | (X[col] > upper)).sum()

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "Lower": lower,
        "Upper": upper,
        "Outliers": outlier_count,
        "Outlier_pct": outlier_count / len(X) * 100,
        "Skewness": X[col].skew()
    })

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table.sort_values("Outlier_pct", ascending=False).round(3))

## 9. Population Stability Index

PSI compares the distribution of a feature between train and test data. A common heuristic is PSI < 0.10 for small shift, 0.10–0.25 for moderate shift, and > 0.25 for substantial shift.

**Purpose:** Compare feature distributions in the labelled and unlabelled datasets to identify possible train-test shifts.

In [ ]:
def psi_numeric(train_series, test_series, bins=10):
    train_values = pd.to_numeric(train_series, errors="coerce").dropna()
    test_values = pd.to_numeric(test_series, errors="coerce").dropna()

    edges = np.unique(
        np.quantile(train_values, np.linspace(0, 1, bins + 1))
    )
    edges[0] = -np.inf
    edges[-1] = np.inf

    train_bin = pd.cut(train_values, bins=edges, include_lowest=True)
    test_bin = pd.cut(test_values, bins=edges, include_lowest=True)

    train_pct = train_bin.value_counts(
        normalize=True, sort=False
    )
    test_pct = test_bin.value_counts(
        normalize=True, sort=False
    ).reindex(train_pct.index, fill_value=0)

    train_pct = train_pct.clip(lower=1e-6)
    test_pct = test_pct.clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi

In [ ]:
def psi_categorical(train_series, test_series):
    train_pct = (
        train_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    test_pct = (
        test_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    categories = train_pct.index.union(test_pct.index)

    train_pct = train_pct.reindex(categories, fill_value=0).clip(lower=1e-6)
    test_pct = test_pct.reindex(categories, fill_value=0).clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi

In [ ]:
psi_rows = []

psi_numeric_cols = [
    col for col in numerical_cols
    if train[col].nunique(dropna=True) > 1
]

for col in psi_numeric_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Numerical",
        "PSI": psi_numeric(train[col], test[col])
    })

for col in categorical_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Categorical",
        "PSI": psi_categorical(train[col], test[col])
    })

psi_table = pd.DataFrame(psi_rows)
psi_table["Shift"] = pd.cut(
    psi_table["PSI"],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=["Small", "Moderate", "Substantial"]
)

display(psi_table.sort_values("PSI", ascending=False).round(4))

## 10. Feature Leakage Tests

Leakage occurs when a feature contains target information that would not be available at the real prediction time. Post-outcome fields such as overdue status, collections, write-offs, or recovery information are common leakage risks in credit problems.

**Purpose:** Review suspiciously predictive or post-outcome fields that might expose information unavailable at prediction time.

In [ ]:
leakage_words = [
    "target",
    "label",
    "bad_debt",
    "default",
    "overdue",
    "delinquent",
    "arrears",
    "collection",
    "writeoff",
    "write_off",
    "chargeoff",
    "charge_off",
    "recovery",
    "dpd"
]

name_leakage_candidates = [
    col for col in X.columns
    if any(word in col.lower() for word in leakage_words)
]

print("Suspicious feature names:")
print(name_leakage_candidates)

In [ ]:
numeric_with_target = train[numerical_cols + [target_col]].copy()

target_correlations = (
    numeric_with_target
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .abs()
    .sort_values(ascending=False)
)

display(target_correlations.to_frame("abs_correlation_with_target").head(20))

In [ ]:
binary_candidates = [
    col for col in X.columns
    if train[col].nunique(dropna=False) <= 2
]

copy_scores = pd.Series({
    col: (
        train[col].astype(str) ==
        train[target_col].astype(str)
    ).mean()
    for col in binary_candidates
}).sort_values(ascending=False)

display(copy_scores.to_frame("target_match_rate").head(20))

## 11. Optional Feature Engineering

**Purpose:** Create additional inputs only when the released columns and business meaning make the transformation appropriate.

### Date Features

Convert a date column into components such as year, month, quarter, and day of week.

**Purpose:** Derive calendar components from a valid date field when those components could represent useful timing patterns.

In [ ]:
date_col = "application_date"

train[date_col] = pd.to_datetime(train[date_col], errors="coerce")
test[date_col] = pd.to_datetime(test[date_col], errors="coerce")

train["year"] = train[date_col].dt.year
train["month"] = train[date_col].dt.month
train["quarter"] = train[date_col].dt.quarter
train["day_of_week"] = train[date_col].dt.dayofweek

test["year"] = test[date_col].dt.year
test["month"] = test[date_col].dt.month
test["quarter"] = test[date_col].dt.quarter
test["day_of_week"] = test[date_col].dt.dayofweek

### Log Transformation

Useful for highly right-skewed positive numerical variables.

**Purpose:** Reduce the scale and skew of a suitable non-negative numeric variable while retaining its row-level information.

In [ ]:
col = "income"

train[col + "_log"] = np.log1p(train[col])
test[col + "_log"] = np.log1p(test[col])

### Ratio Feature

Create a business ratio from two existing variables.

**Purpose:** Represent a meaningful relative quantity from two variables, taking care with zero or missing denominators.

In [ ]:
train["new_ratio"] = train["numerator_col"] / train["denominator_col"]
test["new_ratio"] = test["numerator_col"] / test["denominator_col"]

### Interaction Feature

Represent the combined effect of two numerical variables.

**Purpose:** Represent a possible combined effect of two numeric variables for models that can use the resulting feature.

In [ ]:
train["interaction_feature"] = train["col_a"] * train["col_b"]
test["interaction_feature"] = test["col_a"] * test["col_b"]

### Binning

Convert a continuous variable into interpretable groups.

**Purpose:** Group a continuous value into defined ranges when grouped levels are useful and the cut points are defensible.

In [ ]:
bins = [0, 25, 40, 60, np.inf]
labels = ["<=25", "26-40", "41-60", "60+"]

train["age_group"] = pd.cut(train["age"], bins=bins, labels=labels)
test["age_group"] = pd.cut(test["age"], bins=bins, labels=labels)

## 12. Train and Validation Split

Stratified splitting keeps the target proportion similar in both sets.

**Purpose:** Reserve labelled observations for an honest model comparison while preserving class proportions where appropriate.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(X_train.shape, X_valid.shape)
print(y_train.value_counts(normalize=True).round(3))
print(y_valid.value_counts(normalize=True).round(3))

## 13. Missing Value Methods

**Purpose:** Review alternative imputation rules and choose them within the fitted preprocessing pipeline.

### Median Imputation

Median is less sensitive to extreme values than mean.

**Purpose:** Fill numeric gaps with a robust centre that is less affected by extreme observations.

In [ ]:
numeric_median = SimpleImputer(strategy="median")

### Mean Imputation

Mean is simple and works best when extreme values are limited.

**Purpose:** Fill numeric gaps with the arithmetic average when the distribution and outlier profile make it suitable.

In [ ]:
numeric_mean = SimpleImputer(strategy="mean")

### Most Frequent Imputation

**Purpose:** Fill categorical gaps with the most common observed category.

In [ ]:
categorical_mode = SimpleImputer(strategy="most_frequent")

### Constant Category

**Purpose:** Represent missing categorical values as an explicit level when missingness itself may carry information.

In [ ]:
categorical_unknown = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

## 14. Encoding Methods

**Purpose:** Convert categories into numeric representations that the selected estimator can process.

### One-Hot Encoding

Create one binary column for each category.

**Purpose:** Create indicator columns for categories without assigning them an artificial order.

In [ ]:
onehot = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

### Ordinal Encoding

Convert categories to integer codes. The numeric order does not automatically imply business order.

**Purpose:** Map categories to integer codes; use only when the representation is appropriate because the codes imply an order to many models.

In [ ]:
ordinal = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

## 15. Scaling Methods

**Purpose:** Review transformations that put numeric features on comparable scales for scale-sensitive models.

### StandardScaler

Center variables around zero and scale them by standard deviation.

**Purpose:** Centre numeric features and scale them by their standard deviation.

In [ ]:
standard_scaler = StandardScaler()

### RobustScaler

Scale using median and interquartile range, reducing the effect of extreme values.

**Purpose:** Scale numeric features using the median and interquartile range to reduce sensitivity to extreme values.

In [ ]:
robust_scaler = RobustScaler()

## 16. Preprocessing Pipelines

**Purpose:** Combine transformations so each model receives consistently prepared inputs and transformations are fitted on training data only.

### Standard Scaling + One-Hot Encoding

**Purpose:** Use imputation, numeric scaling, and categorical indicators together for models that benefit from scaled inputs.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess_standard = ColumnTransformer([
    ("num", numeric_pipe, numerical_cols),
    ("cat", categorical_pipe, categorical_cols)
])

### No Scaling + One-Hot Encoding

Suitable for tree-based models that do not require feature scaling.

**Purpose:** Impute missing values and encode categories without scaling numeric values, a common setup for tree models.

In [ ]:
numeric_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess_tree = ColumnTransformer([
    ("num", numeric_pipe_tree, numerical_cols),
    ("cat", categorical_pipe_tree, categorical_cols)
])

### No Scaling + Ordinal Encoding

A compact alternative when one-hot encoding creates too many columns.

**Purpose:** Use compact category codes when their imposed numeric structure is acceptable to the chosen model.

In [ ]:
numeric_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocess_ordinal = ColumnTransformer([
    ("num", numeric_pipe_ordinal, numerical_cols),
    ("cat", categorical_pipe_ordinal, categorical_cols)
])

## 17. Model Evaluation Template

The assessment focuses on F1 measures and predictive ranking. Average Precision is calculated from predicted probabilities.

**Purpose:** Create a consistent place to record validation metrics so candidate models can be compared on the same basis.

In [ ]:
results = []

## 18. Dummy Classifier

A simple benchmark showing how much the real models improve over a naive prediction strategy.

**Purpose:** Establish a simple baseline that shows how much value the feature-based models add over a naive prediction rule.

In [ ]:
dummy = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DummyClassifier(strategy="prior"))
])

dummy.fit(X_train, y_train)

In [ ]:
pred_dummy = dummy.predict(X_valid)
prob_dummy = dummy.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Dummy",
    "F1_0": f1_score(y_valid, pred_dummy, pos_label=0),
    "F1_1": f1_score(y_valid, pred_dummy, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_dummy, average="weighted"),
    "AP": average_precision_score(y_valid, prob_dummy)
})

display(pd.DataFrame(results).tail(1).round(4))

## 19. Logistic Regression

A linear probabilistic classifier. It is fast, interpretable, and benefits from scaled numerical variables.

**Purpose:** Fit a linear probabilistic classifier as an interpretable benchmark for the prepared features.

In [ ]:
logistic = Pipeline([
    ("preprocess", preprocess_standard),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

logistic.fit(X_train, y_train)

In [ ]:
pred_logistic = logistic.predict(X_valid)
prob_logistic = logistic.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Logistic Regression",
    "F1_0": f1_score(y_valid, pred_logistic, pos_label=0),
    "F1_1": f1_score(y_valid, pred_logistic, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_logistic, average="weighted"),
    "AP": average_precision_score(y_valid, prob_logistic)
})

display(pd.DataFrame(results).tail(1).round(4))

## 20. Logistic Regression with Class Weight

Class weighting gives more importance to the minority class without creating new observations.

**Purpose:** Compare a cost-sensitive linear model that gives additional training weight to the less frequent class.

In [ ]:
logistic_balanced = Pipeline([
    ("preprocess", preprocess_standard),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_balanced.fit(X_train, y_train)

In [ ]:
pred_logistic_balanced = logistic_balanced.predict(X_valid)
prob_logistic_balanced = logistic_balanced.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Logistic Balanced",
    "F1_0": f1_score(y_valid, pred_logistic_balanced, pos_label=0),
    "F1_1": f1_score(y_valid, pred_logistic_balanced, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_logistic_balanced, average="weighted"),
    "AP": average_precision_score(y_valid, prob_logistic_balanced)
})

display(pd.DataFrame(results).tail(1).round(4))

## 21. Decision Tree

A rule-based model that captures nonlinear patterns but can overfit.

**Purpose:** Fit a single tree to capture nonlinear splits and inspect the trade-off between flexibility and overfitting.

In [ ]:
decision_tree = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

decision_tree.fit(X_train, y_train)

In [ ]:
pred_tree = decision_tree.predict(X_valid)
prob_tree = decision_tree.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Decision Tree",
    "F1_0": f1_score(y_valid, pred_tree, pos_label=0),
    "F1_1": f1_score(y_valid, pred_tree, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_tree, average="weighted"),
    "AP": average_precision_score(y_valid, prob_tree)
})

display(pd.DataFrame(results).tail(1).round(4))

## 22. Random Forest

An ensemble of decision trees that captures nonlinear relationships and usually reduces overfitting compared with a single tree.

**Purpose:** Fit an ensemble of randomized trees whose averaged predictions can capture nonlinear patterns.

In [ ]:
random_forest = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

random_forest.fit(X_train, y_train)

In [ ]:
pred_rf = random_forest.predict(X_valid)
prob_rf = random_forest.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Random Forest",
    "F1_0": f1_score(y_valid, pred_rf, pos_label=0),
    "F1_1": f1_score(y_valid, pred_rf, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_rf, average="weighted"),
    "AP": average_precision_score(y_valid, prob_rf)
})

display(pd.DataFrame(results).tail(1).round(4))

## 23. Random Forest with Class Weight

A simple imbalance treatment for tree ensembles.

**Purpose:** Compare a weighted forest to see whether class weighting changes minority-class performance.

In [ ]:
random_forest_balanced = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

random_forest_balanced.fit(X_train, y_train)

In [ ]:
pred_rf_balanced = random_forest_balanced.predict(X_valid)
prob_rf_balanced = random_forest_balanced.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Random Forest Balanced",
    "F1_0": f1_score(y_valid, pred_rf_balanced, pos_label=0),
    "F1_1": f1_score(y_valid, pred_rf_balanced, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_rf_balanced, average="weighted"),
    "AP": average_precision_score(y_valid, prob_rf_balanced)
})

display(pd.DataFrame(results).tail(1).round(4))

## 24. Extra Trees

An ensemble similar to Random Forest but with more randomized split selection.

**Purpose:** Fit a highly randomized tree ensemble as another nonlinear candidate for comparison.

In [ ]:
extra_trees = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

extra_trees.fit(X_train, y_train)

In [ ]:
pred_extra = extra_trees.predict(X_valid)
prob_extra = extra_trees.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Extra Trees",
    "F1_0": f1_score(y_valid, pred_extra, pos_label=0),
    "F1_1": f1_score(y_valid, pred_extra, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_extra, average="weighted"),
    "AP": average_precision_score(y_valid, prob_extra)
})

display(pd.DataFrame(results).tail(1).round(4))

## 25. HistGradientBoosting

A boosting model that builds trees sequentially and focuses on correcting previous errors.

**Purpose:** Fit trees sequentially so later trees can improve on errors made by earlier trees.

In [ ]:
hist_gb = Pipeline([
    ("preprocess", preprocess_ordinal),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=250,
        random_state=RANDOM_STATE
    ))
])

hist_gb.fit(X_train, y_train)

In [ ]:
pred_hist = hist_gb.predict(X_valid)
prob_hist = hist_gb.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "HistGradientBoosting",
    "F1_0": f1_score(y_valid, pred_hist, pos_label=0),
    "F1_1": f1_score(y_valid, pred_hist, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_hist, average="weighted"),
    "AP": average_precision_score(y_valid, prob_hist)
})

display(pd.DataFrame(results).tail(1).round(4))

## 26. XGBoost

A gradient boosting tree model with strong regularization and flexible hyperparameters.

Install the package before the competition environment if it is not already available.

**Purpose:** Fit an optional gradient-boosted tree model when the package is available in the working environment.

In [ ]:
# !pip install xgboost
from xgboost import XGBClassifier

In [ ]:
xgb = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

xgb.fit(X_train, y_train)

In [ ]:
pred_xgb = xgb.predict(X_valid)
prob_xgb = xgb.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "XGBoost",
    "F1_0": f1_score(y_valid, pred_xgb, pos_label=0),
    "F1_1": f1_score(y_valid, pred_xgb, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_xgb, average="weighted"),
    "AP": average_precision_score(y_valid, prob_xgb)
})

display(pd.DataFrame(results).tail(1).round(4))

## 27. Compare Models

**Purpose:** Rank completed validation runs using the chosen metrics and use the comparison to support model selection.

In [ ]:
model_results = pd.DataFrame(results)
model_results = model_results.sort_values(
    ["AP", "Weighted_F1"],
    ascending=False
).reset_index(drop=True)

display(model_results.round(4))

## 28. Detailed Result for One Model

**Purpose:** Inspect class-level metrics and plots for the candidate selected for closer validation analysis.

In [ ]:
selected_model = random_forest
selected_pred = pred_rf
selected_prob = prob_rf

In [ ]:
print(classification_report(
    y_valid,
    selected_pred,
    digits=4,
    zero_division=0
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    selected_pred,
    values_format="d"
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_valid,
    selected_prob
)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(alpha=0.3)
plt.show()

## 29. Threshold Tuning

The probability threshold changes the trade-off between false negatives and false positives.

**Purpose:** Compare probability cutoffs to understand how the balance between positive-class precision and recall changes.

In [ ]:
threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    pred_threshold = (selected_prob >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(threshold, 2),
        "Precision_1": precision_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "Recall_1": recall_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "F1_1": f1_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "Weighted_F1": f1_score(
            y_valid,
            pred_threshold,
            average="weighted",
            zero_division=0
        )
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(4))

In [ ]:
best_threshold_row = threshold_table.loc[
    threshold_table["F1_1"].idxmax()
]

display(best_threshold_row.to_frame().T.round(4))

## 30. Random Forest Tuning

Important Random Forest parameters include the number of trees, tree depth, minimum samples per split or leaf, feature sampling, and class weighting.

**Purpose:** Search a defined set of forest settings using cross-validation, then assess the selected configuration on validation data.

In [ ]:
rf_tune_model = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

rf_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [6, 8, 10, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 3, 5],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": [None, "balanced"]
}

In [ ]:
rf_search = RandomizedSearchCV(
    rf_tune_model,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best CV AP:", rf_search.best_score_)
print("Best parameters:")
print(rf_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_

pred_rf_best = rf_best.predict(X_valid)
prob_rf_best = rf_best.predict_proba(X_valid)[:, 1]

print("F1 0:", round(f1_score(y_valid, pred_rf_best, pos_label=0), 4))
print("F1 1:", round(f1_score(y_valid, pred_rf_best, pos_label=1), 4))
print("Weighted F1:", round(f1_score(y_valid, pred_rf_best, average="weighted"), 4))
print("AP:", round(average_precision_score(y_valid, prob_rf_best), 4))

## 31. XGBoost Tuning

Important XGBoost parameters include the number and depth of trees, learning rate, row and feature sampling, minimum child weight, and regularization.

**Purpose:** Search selected boosting settings using cross-validation when XGBoost is installed and runtime permits.

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
class_ratio = negative_count / positive_count

xgb_tune_model = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", XGBClassifier(
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

xgb_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 8],
    "model__reg_alpha": [0, 0.01, 0.1, 0.5],
    "model__reg_lambda": [1, 2, 5, 10],
    "model__scale_pos_weight": [1, class_ratio]
}

In [ ]:
xgb_search = RandomizedSearchCV(
    xgb_tune_model,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print("Best CV AP:", xgb_search.best_score_)
print("Best parameters:")
print(xgb_search.best_params_)

In [ ]:
xgb_best = xgb_search.best_estimator_

pred_xgb_best = xgb_best.predict(X_valid)
prob_xgb_best = xgb_best.predict_proba(X_valid)[:, 1]

print("F1 0:", round(f1_score(y_valid, pred_xgb_best, pos_label=0), 4))
print("F1 1:", round(f1_score(y_valid, pred_xgb_best, pos_label=1), 4))
print("Weighted F1:", round(f1_score(y_valid, pred_xgb_best, average="weighted"), 4))
print("AP:", round(average_precision_score(y_valid, prob_xgb_best), 4))

## 32. Cross-Validation

Stratified K-Fold evaluates the selected pipeline across several balanced folds.

**Purpose:** Estimate how consistently a pipeline performs across multiple stratified folds of the labelled data.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    random_forest,
    X,
    y,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1
)

print("Fold AP:", np.round(cv_scores, 4))
print("Mean AP:", round(cv_scores.mean(), 4))
print("Std AP:", round(cv_scores.std(), 4))

## 33. Tree Feature Importance

Tree importance indicates how strongly transformed features contributed to split decisions.

**Purpose:** Summarize which transformed features the fitted tree ensemble used for its split decisions.

In [ ]:
rf_preprocess = random_forest.named_steps["preprocess"]
rf_model = random_forest.named_steps["model"]

feature_names = rf_preprocess.get_feature_names_out()
feature_importance = rf_model.feature_importances_

importance_table = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values("Importance", ascending=False)

display(importance_table.head(20))

## 34. Logistic Regression Coefficients

Coefficients show the direction and relative magnitude of the linear relationship on the transformed feature scale.

**Purpose:** Inspect coefficient direction and relative magnitude on the transformed feature scale.

In [ ]:
log_preprocess = logistic.named_steps["preprocess"]
log_model = logistic.named_steps["model"]

feature_names_log = log_preprocess.get_feature_names_out()
coefficients = log_model.coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": feature_names_log,
    "Coefficient": coefficients,
    "Abs_Coefficient": np.abs(coefficients)
}).sort_values("Abs_Coefficient", ascending=False)

display(coefficient_table.head(20))

## 35. Permutation Importance

Permutation importance measures how much validation performance decreases after shuffling one raw feature.

**Purpose:** Estimate how validation performance changes when one raw feature's values are shuffled.

In [ ]:
perm = permutation_importance(
    selected_model,
    X_valid,
    y_valid,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

permutation_table = pd.DataFrame({
    "Feature": X_valid.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(permutation_table.head(20))

## 36. Final Model

**Purpose:** Record the candidate pipeline and decision threshold that will be used for the final prediction workflow.

In [ ]:
final_model = random_forest
final_threshold = 0.50

## 37. Refit on Full Training Data

Fit the selected pipeline again using all labelled observations.

**Purpose:** Refit the selected pipeline on all labelled observations after model selection is complete.

In [ ]:
final_model.fit(X, y)

## 38. Predict Test Data

Generate probabilities first, then convert them into binary labels using the selected threshold.

**Purpose:** Generate test-set predictions from the refitted pipeline using the chosen probability threshold.

In [ ]:
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= final_threshold).astype(int)

In [ ]:
prediction_preview = pd.DataFrame({
    "Probability": test_probability[:10],
    "Prediction": test_prediction[:10]
})

display(prediction_preview)
print("Positive rate:", round(test_prediction.mean(), 4))

## 39. Create Submission

**Purpose:** Place predictions into the supplied submission structure and save the required output file.

In [ ]:
prediction_col = "target"

submission[prediction_col] = test_prediction

display(submission.head())
print(submission.shape)
print(submission[prediction_col].value_counts())

In [ ]:
submission.to_csv(
    "submission_final.csv",
    index=False
)

## 40. Result Tables for the Report

**Purpose:** Collect completed validation metrics and final model details for concise reporting.

In [ ]:
display(model_results.round(4))

In [ ]:
final_summary = pd.DataFrame({
    "Item": [
        "Final model",
        "Validation method",
        "Threshold",
        "Train rows",
        "Test rows",
        "Number of features"
    ],
    "Value": [
        "Random Forest",
        "80/20 stratified split",
        final_threshold,
        len(train),
        len(test),
        X.shape[1]
    ]
})

display(final_summary)

## 41. SHAP Analysis

SHAP values explain how individual features contribute to model predictions. Global plots show important drivers and direction of influence; local plots explain one observation. SHAP supports model interpretation but does not establish causation.

**Purpose:** Use SHAP as an optional model-interpretation method; its explanations describe model behaviour, not causation.

### XGBoost SHAP

**Purpose:** Explain the fitted boosting model's global and selected individual predictions using SHAP values.

In [ ]:
# !pip install shap
import shap

In [ ]:
xgb_shap_pipeline = xgb_best
xgb_shap_preprocess = xgb_shap_pipeline.named_steps["preprocess"]
xgb_shap_model = xgb_shap_pipeline.named_steps["model"]

X_shap_sample = X_valid.sample(
    n=min(500, len(X_valid)),
    random_state=RANDOM_STATE
)

X_shap_matrix = xgb_shap_preprocess.transform(X_shap_sample)
xgb_feature_names = xgb_shap_preprocess.get_feature_names_out()

X_shap = pd.DataFrame(
    X_shap_matrix,
    columns=xgb_feature_names,
    index=X_shap_sample.index
)

In [ ]:
xgb_explainer = shap.TreeExplainer(xgb_shap_model)
xgb_shap_values = xgb_explainer(X_shap)

In [ ]:
shap.plots.bar(
    xgb_shap_values,
    max_display=15
)

In [ ]:
shap.plots.beeswarm(
    xgb_shap_values,
    max_display=15
)

In [ ]:
shap.plots.waterfall(
    xgb_shap_values[0],
    max_display=15
)

### Random Forest SHAP

**Purpose:** Explain the fitted forest's global and selected individual predictions using SHAP values.

In [ ]:
rf_shap_pipeline = rf_best
rf_shap_preprocess = rf_shap_pipeline.named_steps["preprocess"]
rf_shap_model = rf_shap_pipeline.named_steps["model"]

X_shap_rf_matrix = rf_shap_preprocess.transform(X_shap_sample)
rf_feature_names = rf_shap_preprocess.get_feature_names_out()

X_shap_rf = pd.DataFrame(
    X_shap_rf_matrix,
    columns=rf_feature_names,
    index=X_shap_sample.index
)

In [ ]:
rf_explainer = shap.TreeExplainer(rf_shap_model)
rf_shap_values = rf_explainer(X_shap_rf)

rf_shap_class1 = rf_shap_values[:, :, 1]

In [ ]:
shap.plots.bar(
    rf_shap_class1,
    max_display=15
)

In [ ]:
shap.plots.beeswarm(
    rf_shap_class1,
    max_display=15
)

In [ ]:
shap.plots.waterfall(
    rf_shap_class1[0],
    max_display=15
)

## 42. Report Outline

### Best model architecture and workflow
- Data preparation
- Missing-value treatment
- Encoding
- Scaling
- Feature engineering
- Class imbalance treatment
- Final algorithm
- Main hyperparameters
- Validation strategy
- Leakage prevention
- Final refit

### Results analysis
- F1 for label 0
- F1 for label 1
- Weighted F1
- Average Precision
- Confusion matrix
- Precision-recall trade-off
- Model comparison
- False negative implication
- False positive implication
- Limitation
- Realistic improvement
- Feature importance

**Purpose:** Organize the final report around the selected workflow, validation evidence, interpretation, and limitations.